# scConcept Colab Quick Start

This notebook provides a minimal, step-by-step example for running scConcept in Google Colab.

The workflow includes:

1. Installing scConcept
2. Checking GPU availability
3. Uploading input data
4. Processing data into the required `.mat` format
5. Extracting topics
6. Generating biological concepts using an LLM
7. Annotating cells by concepts

Before running this notebook, please prepare:

- a single-cell expression matrix, such as `your_count_matrix.csv`
- a cell annotation file, such as `your_cell_label.csv`
- an OpenAI API key

In [ ]:
# ============================================================
# Cell 1. Install scConcept
# ============================================================

!git clone https://github.com/li-lab-mcgill/scConcept.git
%cd scConcept

# Install PyTorch with CUDA support for GPU acceleration
!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 \
  --index-url https://download.pytorch.org/whl/cu128

# Install other dependencies
!pip install -r requirements.txt

In [ ]:
# ============================================================
# Cell 2. Check GPU
# ============================================================

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    device = "cuda:0"
else:
    print("No GPU detected. Using CPU.")
    device = "cpu"

print("Using device:", device)

In [ ]:
# ============================================================
# Cell 3. Provide OpenAI API key
# ============================================================

# Replace this with your own API key.
# Do NOT share notebooks containing your private API key.
api_key = "YOUR_API_KEY"

In [ ]:
# ============================================================
# Cell 4. Initialize scConcept
# ============================================================

from scConcept import ScConcept

scconcept = ScConcept()

In [ ]:
# ============================================================
# Cell 5. Choose data mode
# ============================================================

# Option A: use the default preprocessed demo dataset.
# The repository should contain:
#   Datasets/pollen.mat
#
# Option B: upload your own raw/count data and process it.

USE_DEMO_DATA = True

if USE_DEMO_DATA:
    dataset_name = "pollen"
    print(f"Using demo dataset: {dataset_name}")
    print("Expected file: Datasets/pollen.mat")

else:
    dataset_name = "your_dataset"
    print("Using custom dataset mode.")

In [ ]:
# ============================================================
# Cell 6A. Demo mode: directly use pollen.mat
# ============================================================

# If USE_DEMO_DATA=True, no preprocessing is needed.
# The topic model will directly read:
#   Datasets/pollen.mat

if USE_DEMO_DATA:
    import os

    demo_path = "Datasets/pollen.mat"
    if not os.path.exists(demo_path):
        raise FileNotFoundError(
            "Demo dataset not found. Please make sure Datasets/pollen.mat exists."
        )

    print("Demo dataset found:", demo_path)

# # ============================================================
# # Cell 6B. Custom mode: upload and process your own data
# # ============================================================

# # Run this cell only when USE_DEMO_DATA=False.
# #
# # Supported inputs:
# # 1. CSV expression matrix + CSV label file
# # 2. AnnData .h5ad file
# #
# # The uploaded files will be moved to:
# #   Datasets/

# if not USE_DEMO_DATA:
#     from google.colab import files

#     uploaded = files.upload()

#     !mkdir -p Datasets
#     !mv * Datasets/

#     print("Uploaded files moved to Datasets/")

In [ ]:
# ============================================================
# Cell 7. Process custom data if needed
# ============================================================

# Run this cell only when USE_DEMO_DATA=False.
#
# Example 1: CSV input
# - input_name: expression matrix file in Datasets/
# - label_name: cell label file in Datasets/
# - output_name: dataset name used in downstream steps
#
# Example 2: h5ad input
# - input_name: h5ad file in Datasets/
# - label_key: label column in adata.obs

if not USE_DEMO_DATA:
    # ---- CSV example ----
    mat_file = scconcept.dataprocess(
        input_name="your_count_matrix.csv",
        label_name="your_cell_label.csv",
        output_name=dataset_name
    )

    # ---- h5ad example ----
    # mat_file = scconcept.dataprocess(
    #     input_name="your_dataset.h5ad",
    #     output_name=dataset_name,
    #     label_key="cell_type"
    # )

    print("Processed .mat file:", mat_file)

In [ ]:
# ============================================================
# Cell 8. Topic extraction
# ============================================================

# This step trains ECRTM and extracts topic gene lists.
#
# For a quick Colab test, use fewer epochs such as 50 or 100.
# For formal experiments, use epochs=500.

topic_file = scconcept.topic(
    dataset=dataset_name,
    n_topic=50,
    epochs=500,
    batch_size=512,
    device=device
)

print("Topic file:", topic_file)

In [ ]:
# ============================================================
# Cell 9. Concept generation
# ============================================================

# This step uses GPT to distill topic gene lists into biological concepts.
#
# Output:
#   Results/<dataset_name>/<dataset_name>_concepts.json

concepts = scconcept.concept(
    topic_file=topic_file,
    api_key=api_key,
    model="gpt-5"
)

print("num concepts:", len(concepts))
print("first concept:", concepts[0])

In [ ]:
# ============================================================
# Cell 10. Cell annotation
# ============================================================

# This step assigns each cell to a concept using concept gene expression.
#
# Output:
#   concept scores
#   concept names
#   predicted concept label for each cell

scores, concept_names, pred_labels = scconcept.annotation(
    concepts=concepts,
    dataset=dataset_name,
    topk=100
)

print("scores shape:", scores.shape)
print("concept names:", concept_names)
print("first 10 predicted labels:", pred_labels[:10])

In [ ]:
# ============================================================
# Cell 11. Evaluation and visualization
# ============================================================

# Evaluation requires cell labels stored in the .mat file.
# species can be "human" or "mouse".

metrics = scconcept.evaluation(
    concepts=concepts,
    dataset=dataset_name,
    species="human"
)

print("metrics:", metrics)

# PCA + UMAP visualization
adata_vis = scconcept.visualization(
    dataset=dataset_name
)

print("UMAP done.")